[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HannahPinson/tue-deeplearning-2AMM10/blob/main/practicals/P5.2_seq2seq_translation.ipynb)

In [ ]:
# If needed, uncomment the next line to install the required packages.
# !pip install -q datasets matplotlib

# Practical 5.2: Sequence-to-Sequence Translation with RNNs and Transformers

In this practical we will:

1. load a bilingual translation dataset from the Hugging Face Hub,
2. preprocess text data and create source/target vocabularies,
3. implement a plain RNN encoder-decoder for machine translation,
4. implement a Transformer encoder-decoder for the same task,
5. generate translations using greedy decoding,


The goal is not to build a production-quality translation system. The goal is to understand how different sequence-to-sequence architectures model dependencies between an input sequence and an output sequence.

In [1]:
import math
import random
import re
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import numpy as np

from datasets import load_dataset

c:\Users\trist\Documents\GitHub\tue-deeplearning-2AMM10\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)
device = torch.device("cpu")

Using device: cuda


## 0. What is neural machine translation?

Machine translation is a **sequence-to-sequence** problem. Given a source sentence

$$
\mathbf{x} = (x_1, x_2, \dots, x_T)
$$

in one language, the model must generate a target sentence

$$
\mathbf{y} = (y_1, y_2, \dots, y_S)
$$

in another language.

A neural translation model predicts the target sentence one token at a time. At decoding step $t$, the model predicts the next target token $y_t$ using:

$$
p(y_t \mid x_1, x_2, \dots, x_T, y_1, y_2, \dots, y_{t-1}).
$$

In words, the model predicts the next translated word using the full source sentence and the target words that have already been generated.

In this practical, we compare two sequence-to-sequence architectures for translation:

1. a plain RNN encoder-decoder;
2. a Transformer encoder-decoder.

## 1. Load a translation dataset

We use the OPUS Books dataset from the Hugging Face Hub. Each datapoint contains a translation pair.

To keep this practical fast, we only use a small subset of the dataset.

In [23]:
dataset_name = "Helsinki-NLP/opus_books"
language_pair = "en-fr"

raw_dataset = load_dataset(dataset_name, language_pair, split="train")
raw_dataset = raw_dataset.shuffle(seed=seed)

# Small subsets keep the practical fast.
num_train_examples = 6000
num_test_examples = 500
num_total = num_train_examples + num_test_examples

raw_dataset = raw_dataset.select(range(num_total))

train_raw = raw_dataset.select(range(num_train_examples))
test_raw = raw_dataset.select(range(num_train_examples, num_total))

print(train_raw)
print(test_raw)

Dataset({
    features: ['id', 'translation'],
    num_rows: 6000
})
Dataset({
    features: ['id', 'translation'],
    num_rows: 500
})


In [24]:
# Inspect a few translation pairs.
for i in range(3):
    example = train_raw[i]["translation"]
    print("English:", example["en"])
    print("French :", example["fr"])
    print()

English: "Not if it were my own brother!" cried d’Artagnan, as if carried away by his enthusiasm.
French : «Non, fût-ce mon frère!» s'écria d'Artagnan comme emporté par l'enthousiasme.

English: "Mr. Conseil put one over on me!"
French : Monsieur Conseil qui me fait poser !

English: What sister would think herself at liberty to do it, unless there were something very objectionable? If they believed him attached to me, they would not try to part us; if he were so, they could not succeed.
French : Cependant si elles croyaient qu’il m’aime, elles ne chercheraient pas a nous séparer, et, s’il m’aimait, elles ne pourraient y réussir.



## 2. Text preprocessing and vocabularies

Neural networks do not operate directly on strings. Before a sentence can be passed to a neural network, it must be converted into a sequence of integer token IDs.

This usually involves two steps:

1. **Tokenization:** split a sentence into smaller units called tokens.
2. **Numericalization:** map each token to an integer ID using a vocabulary.

Modern language models usually use **subword tokenizers**, such as Byte-Pair Encoding (BPE), WordPiece, or SentencePiece. These tokenizers can split rare words into smaller pieces and reduce the number of unknown tokens.

In this practical, we use a simpler whitespace-and-punctuation tokenizer. This tokenizer is not as strong as modern subword tokenizers, but it is transparent and sufficient for understanding sequence-to-sequence modelling.

We define four special tokens:

```text
<pad> : padding token used to create equal-length batches
<unk> : unknown token for words outside the vocabulary
<bos> : beginning-of-sentence token
<eos> : end-of-sentence token

### TODO: Implement `tokens_to_ids` function

In [25]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]


def tokenize_text(text: str) -> list[str]:
    """
    Convert a sentence string into a list of tokens.

    This tokenizer is intentionally simple. It:
    1. lowercases the input text;
    2. removes leading and trailing whitespace;
    3. separates words from punctuation.

    Parameters
    ----------
    text : str
        Input sentence.

    Returns
    -------
    list[str]
        List of token strings.

    Example
    -------
    >>> tokenize_text("Hello, world!")
    ['hello', ',', 'world', '!']
    """
    text = text.lower().strip()
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)


def build_vocabulary(
    tokenized_sentences: list[list[str]],
    max_vocab_size: int = 8000,
    min_frequency: int = 2,
) -> tuple[dict[str, int], dict[int, str]]:
    """
    Build a vocabulary from tokenized sentences.

    The vocabulary maps tokens to integer IDs. The first IDs are reserved for
    special tokens:

    <pad> -> 0
    <unk> -> 1
    <bos> -> 2
    <eos> -> 3

    Only tokens that appear at least `min_frequency` times are included.
    The vocabulary is capped at `max_vocab_size` tokens, including the special
    tokens.

    Parameters
    ----------
    tokenized_sentences : list[list[str]]
        List of tokenized sentences. Each sentence is represented as a list of
        token strings.
    max_vocab_size : int, default=8000
        Maximum number of tokens in the vocabulary, including special tokens.
    min_frequency : int, default=2
        Minimum frequency required for a token to be included.

    Returns
    -------
    token_to_id : dict[str, int]
        Dictionary mapping each token string to its integer ID.
    id_to_token : dict[int, str]
        Dictionary mapping each integer ID back to its token string.
    """
    token_counts = Counter()

    for tokens in tokenized_sentences:
        token_counts.update(tokens)

    token_to_id = {token: token_id for token_id, token in enumerate(SPECIAL_TOKENS)}

    id_to_token = {token_id: token for token_id, token in enumerate(SPECIAL_TOKENS)}

    for token, count in token_counts.most_common():
        if count < min_frequency:
            continue

        if token in token_to_id:
            continue

        if len(token_to_id) >= max_vocab_size:
            break

        token_id = len(token_to_id)
        token_to_id[token] = token_id
        id_to_token[token_id] = token

    return token_to_id, id_to_token


def ids_to_tokens(
    token_ids: list[int] | torch.Tensor,
    id_to_token: dict[int, str],
) -> list[str]:
    """
    Convert a sequence of token IDs back into token strings.

    This function keeps special tokens such as <pad>, <bos>, and <eos>. This is
    useful for debugging because students can see exactly what the model input
    or output sequence contains.

    Parameters
    ----------
    token_ids : list[int] or torch.Tensor
        Sequence of integer token IDs.
    id_to_token : dict[int, str]
        Vocabulary mapping from integer IDs to token strings.

    Returns
    -------
    list[str]
        List of token strings.
    """
    tokens = []

    for token_id in token_ids:
        token_id = int(token_id)
        token = id_to_token.get(token_id, UNK_TOKEN)
        tokens.append(token)

    return tokens


def tokens_to_text(tokens: list[str]) -> str:
    """
    Convert a list of tokens into a readable sentence.

    This function removes special tokens. It is mainly used when displaying
    generated translations.

    Parameters
    ----------
    tokens : list[str]
        List of token strings.

    Returns
    -------
    str
        Readable sentence string.
    """
    clean_tokens = []

    for token in tokens:
        if token == EOS_TOKEN:
            break

        if token in {PAD_TOKEN, BOS_TOKEN}:
            continue

        clean_tokens.append(token)

    return " ".join(clean_tokens)


def tokens_to_ids(
    tokens: list[str],
    token_to_id: dict[str, int],
    add_bos: bool = False,
    add_eos: bool = False,
) -> list[int]:
    """
    Convert a list of tokens into a list of integer token IDs.
    """

    # If add_bos is True, append BOS_ID at the beginning.
    token_ids = [BOS_ID] if add_bos else []

    # Convert each token to its integer ID.
    # If a token is not in the vocabulary, use UNK_ID.
    for token in tokens:
        token_id = token_to_id.get(token, UNK_ID)
        token_ids.append(token_id)

    # If add_eos is True, append EOS_ID at the end.
    if add_eos:
        token_ids.append(EOS_ID)

    return token_ids

In [26]:
# Tokenize training sentences and build vocabularies only from the training split.
train_source_tokens = [
    tokenize_text(example["translation"]["en"]) for example in train_raw
]

train_target_tokens = [
    tokenize_text(example["translation"]["fr"]) for example in train_raw
]

source_token_to_id, source_id_to_token = build_vocabulary(
    train_source_tokens,
    max_vocab_size=8000,
    min_frequency=2,
)

target_token_to_id, target_id_to_token = build_vocabulary(
    train_target_tokens,
    max_vocab_size=8000,
    min_frequency=2,
)

print("Source vocabulary size:", len(source_id_to_token))
print("Target vocabulary size:", len(target_id_to_token))

print("Example source tokens:", train_source_tokens[0])
print(
    "Example source token IDs:",
    tokens_to_ids(train_source_tokens[0], source_token_to_id),
)

print("Example target tokens:", train_target_tokens[0])
print(
    "Example target token IDs:",
    tokens_to_ids(train_target_tokens[0], target_token_to_id),
)

Source vocabulary size: 6789
Target vocabulary size: 7459
Example source tokens: ['"', 'not', 'if', 'it', 'were', 'my', 'own', 'brother', '!', '"', 'cried', 'd', '’', 'artagnan', ',', 'as', 'if', 'carried', 'away', 'by', 'his', 'enthusiasm', '.']
Example source token IDs: [10, 29, 68, 17, 52, 32, 178, 483, 34, 10, 199, 77, 30, 179, 4, 25, 68, 374, 160, 46, 21, 2016, 6]
Example target tokens: ['«', 'non', ',', 'fût', '-', 'ce', 'mon', 'frère', '!', '»', 's', "'", 'écria', 'd', "'", 'artagnan', 'comme', 'emporté', 'par', 'l', "'", 'enthousiasme', '.']
Example target token IDs: [65, 125, 4, 304, 8, 35, 68, 487, 30, 69, 36, 6, 238, 20, 6, 160, 66, 1725, 55, 15, 6, 2320, 5]


### Source and target vocabularies

In this practical, we build two separate vocabularies:

- a **source vocabulary** for English input tokens;
- a **target vocabulary** for French output tokens.

The encoder receives English sentences, so it uses the source vocabulary to map English tokens to integer IDs. These IDs are then converted into embeddings by the encoder embedding layer.

The decoder receives French tokens during training, starting with the `<bos>` token. Therefore, it uses the target vocabulary to map French tokens to integer IDs, which are then converted into embeddings by the decoder embedding layer.


The integer IDs are vocabulary-specific. For example, ID `42` may correspond to one English token in the source vocabulary and to a different French token in the target vocabulary. This is not a problem because the encoder and decoder use separate vocabularies and separate embedding layers.

Shared vocabularies are also possible. They are often useful in bidirectional or multilingual translation systems, where the same model may translate between the same languages in multiple directions.

## 3. Translation Dataset

We now define a PyTorch `Dataset` for the translation examples.

The role of the dataset is to process **one example at a time**. Each raw example contains an English source sentence and a French target sentence. The dataset will:

1. tokenize the English and French sentences;
2. convert the tokens to integer IDs;
3. return the source sequence, the decoder input sequence, and the decoder output sequence.

### Source sequence

The source sentence is fully observed by the encoder. Therefore, in this practical, we do not add `<bos>` or `<eos>` tokens to the source sequence.

If the source sentence is

$$
(x_1, x_2, \dots, x_T),
$$

then the source sequence stored by the dataset is simply:

```text
source_ids = x_1, x_2, ..., x_T
```

### Target input and target output sequences

The target sentence is different because it must be generated by the decoder. We therefore construct two target-side sequences before padding:

```text
target_input_ids  = <bos>, y_1, y_2, ..., y_S
target_output_ids = y_1,   y_2, ..., y_S, <eos>
```

The `<bos>` token is needed because the decoder must receive an initial input before it can predict the first translated word.

The `<eos>` token is needed because the model must learn when a generated translation should stop.

If the target sentence is

$$
(y_1, y_2, \dots, y_S),
$$

then the decoder receives `target_input_ids`, and the loss is computed against `target_output_ids`.

This means that at each decoder step, the model learns to predict the next target token:

```text
given <bos>              -> predict y_1
given <bos>, y_1         -> predict y_2
...
given <bos>, ..., y_S    -> predict <eos>
```

Creating `target_input_ids` and `target_output_ids` inside the dataset keeps the training loop simple and avoids feeding `<eos>` to the decoder as an input token.

### TODO: Implement the `TranslationDataset` `__getitem__` method

In [27]:
class TranslationDataset(Dataset):
    """
    PyTorch Dataset for sequence-to-sequence translation.

    Each item in the dataset is one translation pair. The dataset converts the
    raw text pair into token ID sequences.

    The source sequence is used by the encoder:

        source_ids = x_1, x_2, ..., x_T

    The target sequence is split into two shifted sequences before padding:

        target_input_ids  = <bos>, y_1, y_2, ..., y_S
        target_output_ids = y_1,   y_2, ..., y_S, <eos>
    """

    def __init__(
        self,
        hf_dataset,
        source_token_to_id: dict[str, int],
        target_token_to_id: dict[str, int],
        source_language: str = "en",
        target_language: str = "fr",
    ) -> None:
        self.dataset = hf_dataset
        self.source_token_to_id = source_token_to_id
        self.target_token_to_id = target_token_to_id
        self.source_language = source_language
        self.target_language = target_language

    def __len__(self) -> int:
        """
        Return the number of translation examples in the dataset.
        """
        return len(self.dataset)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor | str]:
        """
        Convert one raw translation example into token ID tensors.
        """
        translation_pair = self.dataset[index]["translation"]

        source_text = translation_pair[self.source_language]
        target_text = translation_pair[self.target_language]

        source_tokens = tokenize_text(source_text)
        target_tokens = tokenize_text(target_text)

        # TODO: Convert the source tokens to IDs.
        # The source sequence should not contain <bos> or <eos>.
        source_ids = [self.source_token_to_id.get(token, UNK_ID) for token in source_tokens]

        # TODO: Convert the target tokens to decoder input IDs.
        # The target input should be:
        # <bos>, y_1, y_2, ..., y_S
        target_input_ids = [BOS_ID] + [self.target_token_to_id.get(token, UNK_ID) for token in target_tokens]

        # TODO: Convert the target tokens to decoder output IDs.
        # The target output should be:
        # y_1, y_2, ..., y_S, <eos>
        target_output_ids = [self.target_token_to_id.get(token, UNK_ID) for token in target_tokens] + [EOS_ID]

        return {
            "source_ids": torch.tensor(source_ids, dtype=torch.long),
            "target_input_ids": torch.tensor(target_input_ids, dtype=torch.long),
            "target_output_ids": torch.tensor(target_output_ids, dtype=torch.long),
            "source_text": source_text,
            "target_text": target_text,
        }


train_dataset = TranslationDataset(
    hf_dataset=train_raw,
    source_token_to_id=source_token_to_id,
    target_token_to_id=target_token_to_id,
)

test_dataset = TranslationDataset(
    hf_dataset=test_raw,
    source_token_to_id=source_token_to_id,
    target_token_to_id=target_token_to_id,
)


train_dataset = TranslationDataset(
    hf_dataset=train_raw,
    source_token_to_id=source_token_to_id,
    target_token_to_id=target_token_to_id,
)

test_dataset = TranslationDataset(
    hf_dataset=test_raw,
    source_token_to_id=source_token_to_id,
    target_token_to_id=target_token_to_id,
)

Let's inspect one example from `TranslationDataset`. Notice that the source sequence has no boundary tokens, while the target side is split into `target_input_ids` and `target_output_ids`.

In [28]:
sample = train_dataset[0]

source_ids = sample["source_ids"]
target_input_ids = sample["target_input_ids"]
target_output_ids = sample["target_output_ids"]

print("Source text:")
print(sample["source_text"])

print("\nSource IDs:")
print(source_ids.tolist())

print("\nSource IDs -> tokens:")
print(ids_to_tokens(source_ids, source_id_to_token))

print("\nTarget text:")
print(sample["target_text"])

print("\nTarget input IDs:")
print(target_input_ids.tolist())

print("\nTarget input IDs -> tokens:")
print(ids_to_tokens(target_input_ids, target_id_to_token))

print("\nTarget output IDs:")
print(target_output_ids.tolist())

print("\nTarget output IDs -> tokens:")
print(ids_to_tokens(target_output_ids, target_id_to_token))

print("\nTarget input length:", len(target_input_ids))
print("Target output length:", len(target_output_ids))

Source text:
"Not if it were my own brother!" cried d’Artagnan, as if carried away by his enthusiasm.

Source IDs:
[10, 29, 68, 17, 52, 32, 178, 483, 34, 10, 199, 77, 30, 179, 4, 25, 68, 374, 160, 46, 21, 2016, 6]

Source IDs -> tokens:
['"', 'not', 'if', 'it', 'were', 'my', 'own', 'brother', '!', '"', 'cried', 'd', '’', 'artagnan', ',', 'as', 'if', 'carried', 'away', 'by', 'his', 'enthusiasm', '.']

Target text:
«Non, fût-ce mon frère!» s'écria d'Artagnan comme emporté par l'enthousiasme.

Target input IDs:
[2, 65, 125, 4, 304, 8, 35, 68, 487, 30, 69, 36, 6, 238, 20, 6, 160, 66, 1725, 55, 15, 6, 2320, 5]

Target input IDs -> tokens:
['<bos>', '«', 'non', ',', 'fût', '-', 'ce', 'mon', 'frère', '!', '»', 's', "'", 'écria', 'd', "'", 'artagnan', 'comme', 'emporté', 'par', 'l', "'", 'enthousiasme', '.']

Target output IDs:
[65, 125, 4, 304, 8, 35, 68, 487, 30, 69, 36, 6, 238, 20, 6, 160, 66, 1725, 55, 15, 6, 2320, 5, 3]

Target output IDs -> tokens:
['«', 'non', ',', 'fût', '-', 'ce', 'mo

## 4. Collate function and DataLoader

The `TranslationDataset` processes one example at a time. However, neural networks are usually trained on mini-batches of examples.

Translation examples have variable lengths. Different source sentences can have different numbers of tokens, and different target sentences can also have different numbers of tokens.

However, tensors in the same mini-batch must have the same shape. The role of the `collate_fn` is to take a list of examples from the dataset and combine them into padded mini-batch tensors.

In our case, the `collate_fn` pads three sequences:

```text
source_ids
target_input_ids
target_output_ids
```

These sequences are padded separately using the `<pad>` token. This means that source sentences are padded to the length of the longest source sentence in the batch, while target input and target output sequences are padded to the length of the longest target sequence in the batch.

For example, a batch of target input sequences may look like this after padding:

```text
<bos>, je, suis, ici
<bos>, bonjour, <pad>, <pad>
```

The corresponding target output sequences would be:

```text
je,      suis,  ici,   <eos>
bonjour, <eos>, <pad>, <pad>
```

Padding tokens do not correspond to real words. Therefore, they should be ignored when computing the loss.

### TODO: Implement the `collate_translation_batch` function

In [29]:
def pad_sequence_list(
    sequences: list[torch.Tensor],
    pad_value: int = PAD_ID,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Pad a list of 1D token ID tensors to a single 2D tensor.

    Each input sequence may have a different length. This function pads all
    sequences to the length of the longest sequence in the list.

    Parameters
    ----------
    sequences : list[torch.Tensor]
        List of 1D tensors. Each tensor has shape (sequence_length,).
    pad_value : int, default=PAD_ID
        Token ID used for padding shorter sequences.

    Returns
    -------
    padded_sequences : torch.Tensor
        Tensor of shape (batch_size, max_sequence_length), where shorter
        sequences are padded with `pad_value`.
    sequence_lengths : torch.Tensor
        Tensor of shape (batch_size,), containing the original length of each
        sequence before padding.
    """
    batch_size = len(sequences)
    max_sequence_length = max(sequence.numel() for sequence in sequences)

    padded_sequences = torch.full(
        size=(batch_size, max_sequence_length),
        fill_value=pad_value,
        dtype=torch.long,
    )

    sequence_lengths = torch.zeros(batch_size, dtype=torch.long)

    for index, sequence in enumerate(sequences):
        sequence_length = sequence.numel()
        padded_sequences[index, :sequence_length] = sequence
        sequence_lengths[index] = sequence_length

    return padded_sequences, sequence_lengths


def collate_translation_batch(
    batch: list[dict[str, torch.Tensor | str]],
) -> dict[str, torch.Tensor | list[str]]:
    """
    Collate variable-length translation examples into a padded mini-batch.

    The Dataset returns one example at a time. Each example contains:
    - a source sequence for the encoder;
    - a target input sequence for the decoder;
    - a target output sequence used as the prediction target.

    Since different examples can have different lengths, this function pads
    these sequences separately.
    """
    # TODO: Pad the source ID sequences.
    # Use the "source_ids" field from each example in the batch.
    source_ids, source_lengths = pad_sequence_list([example["source_ids"] for example in batch])

    # TODO: Pad the target input ID sequences.
    # Use the "target_input_ids" field from each example in the batch.
    target_input_ids, target_lengths = pad_sequence_list([example["target_input_ids"] for example in batch])

    # TODO: Pad the target output ID sequences.
    # Use the "target_output_ids" field from each example in the batch.
    target_output_ids, _ =  pad_sequence_list([example["target_output_ids"] for example in batch])

    return {
        "source_ids": source_ids,
        "source_lengths": source_lengths,
        "target_input_ids": target_input_ids,
        "target_output_ids": target_output_ids,
        "target_lengths": target_lengths,
        "source_text": [example["source_text"] for example in batch],
        "target_text": [example["target_text"] for example in batch],
    }  # type: ignore

In [30]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_translation_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_translation_batch,
)

Let's take a look at how our `DataLoader` pads source sequences, target input sequences, and target output sequences.

In [31]:
batch = next(iter(train_loader))

print("Batch keys:")
print(batch.keys())

print("\nSource batch shape:")
print(batch["source_ids"].shape)

print("\nTarget input batch shape:")
print(batch["target_input_ids"].shape)

print("\nTarget output batch shape:")
print(batch["target_output_ids"].shape)

print("\nSource lengths:")
print(batch["source_lengths"][:5])

print("\nTarget lengths:")
print(batch["target_lengths"][:5])

Batch keys:
dict_keys(['source_ids', 'source_lengths', 'target_input_ids', 'target_output_ids', 'target_lengths', 'source_text', 'target_text'])

Source batch shape:
torch.Size([32, 100])

Target input batch shape:
torch.Size([32, 114])

Target output batch shape:
torch.Size([32, 114])

Source lengths:
tensor([14, 30,  5, 10, 10])

Target lengths:
tensor([15, 27,  8, 13, 10])


Padding in action:

In [32]:
sample_index = 0

print("Source text:")
print(batch["source_text"][sample_index])

print("\nPadded source tokens:")
print(ids_to_tokens(batch["source_ids"][sample_index], source_id_to_token))

print("\nTarget text:")
print(batch["target_text"][sample_index])

print("\nPadded target input tokens:")
print(ids_to_tokens(batch["target_input_ids"][sample_index], target_id_to_token))

print("\nPadded target output tokens:")
print(ids_to_tokens(batch["target_output_ids"][sample_index], target_id_to_token))

Source text:
It took my breath away, in a manner of . . ."

Padded source tokens:
['it', 'took', 'my', 'breath', 'away', ',', 'in', 'a', 'manner', 'of', '.', '.', '.', '"', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']

Target text:
Cela me coupait, comme on dit, le respi

## 5. Plain RNN encoder-decoder

The following diagram illustrates the plain sequence-to-sequence RNN architecture used in this practical.

<p align="left">
  <img 
    src="https://raw.githubusercontent.com/HannahPinson/tue-deeplearning-2AMM10/main/img/rnn_translation.png" 
    alt="Plain RNN encoder-decoder for machine translation" 
    width="860"
  />
</p> 

A plain RNN encoder reads the source sentence token by token. At each time step, it receives the embedding of the current source token and updates its hidden state:

$$
\mathbf{h}_t = \mathrm{RNN}_{enc}(\mathbf{e}(x_t), \mathbf{h}_{t-1}).
$$

After the last source token, the final encoder hidden state is used as a fixed-size representation of the whole source sentence:

$$
\mathbf{s}_0 = \mathbf{h}_T.
$$

This final encoder hidden state initializes the decoder. The decoder is another RNN. It receives the target input sequence and predicts the target output sequence.

In our dataset, the target side has already been represented as two shifted sequences:

```text
target_input_ids  = <bos>, y_1, y_2, ..., y_S
target_output_ids = y_1,   y_2, ..., y_S, <eos>
```

The decoder receives `target_input_ids`. At each time step, a linear classifier is applied to the decoder hidden state to predict a distribution over the target vocabulary. The loss is computed by comparing these predictions to `target_output_ids`.

This architecture is simple, but it has an important limitation: the whole source sentence must be compressed into a single hidden vector. This is the **encoder bottleneck**.

### TODO: Implement the Seq2Seq RNN

In [33]:
class PlainRNNEncoder(nn.Module):
    """
    Plain RNN encoder for the source sentence.

    The encoder receives a padded batch of source token IDs and produces the
    final hidden state of the RNN. This final hidden state is used as a
    fixed-size representation of the whole source sentence.
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID,
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
            dropout=dropout if num_layers > 1 else 0.0,
            
        )

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
    ) -> torch.Tensor:
        """
        Encode a padded batch of source sequences.

        Parameters
        ----------
        source_ids : torch.Tensor
            Source token IDs of shape (batch_size, source_length).
        source_lengths : torch.Tensor
            Original source lengths before padding. Shape: (batch_size,).

        Returns
        -------
        torch.Tensor
            Final encoder hidden state of shape
            (num_layers, batch_size, hidden_dim).
        """
        # Convert source token IDs to embeddings.
        # Shape should become:
        # (batch_size, source_length, embedding_dim)
        embedded_source = self.embedding(source_ids)

        # Pack the padded sequence so that the RNN ignores padding tokens.
        # Hint: use nn.utils.rnn.pack_padded_sequence.
        # Remember to move source_lengths to CPU and set batch_first=True.
        packed_source = nn.utils.rnn.pack_padded_sequence(
            embedded_source,
            source_lengths.cpu().tolist(),
            batch_first=True,
            enforce_sorted=False,
        )

        # Run the encoder RNN.
        # We only need the final hidden state.
        _, hidden = self.rnn(packed_source)

        return hidden


class PlainRNNDecoder(nn.Module):
    """
    Plain RNN decoder for the target sentence.

    The decoder receives the target input sequence and an initial hidden state
    from the encoder. At each target position, it predicts a distribution over
    the target vocabulary.
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID,
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.output_layer = nn.Linear(
            in_features=hidden_dim,
            out_features=vocab_size,
        )

    def forward(
        self,
        target_input_ids: torch.Tensor,
        hidden: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Decode a padded batch of target input sequences.

        Parameters
        ----------
        target_input_ids : torch.Tensor
            Target input token IDs of shape (batch_size, target_length).
        hidden : torch.Tensor
            Initial decoder hidden state from the encoder.
            Shape: (num_layers, batch_size, hidden_dim).

        Returns
        -------
        logits : torch.Tensor
            Unnormalized scores over the target vocabulary.
            Shape: (batch_size, target_length, vocab_size).
        hidden : torch.Tensor
            Final decoder hidden state.
            Shape: (num_layers, batch_size, hidden_dim).
        """
        # Convert target input token IDs to embeddings.
        # Shape should become:
        # (batch_size, target_length, embedding_dim)
        embedded_target = self.embedding(target_input_ids)

        # TODO: Run the decoder RNN.
        # The decoder should be initialized with the encoder hidden state.
        decoder_outputs, hidden = self.rnn(embedded_target, hidden)

        # TODO: Convert decoder outputs to vocabulary logits.
        # Shape should become:
        # (batch_size, target_length, vocab_size)
        logits = self.output_layer(decoder_outputs)

        return logits, hidden


class PlainRNNSeq2Seq(nn.Module):
    """
    Plain RNN encoder-decoder model for machine translation.

    The model first encodes the full source sentence using the encoder RNN.
    The final encoder hidden state is then used to initialize the decoder RNN.
    The decoder receives the target input sequence and predicts the next target
    token at each position.
    """

    def __init__(
        self,
        source_vocab_size: int,
        target_vocab_size: int,
        embedding_dim: int = 128,
        hidden_dim: int = 256,
        num_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()

        self.encoder = PlainRNNEncoder(
            vocab_size=source_vocab_size,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
        )

        self.decoder = PlainRNNDecoder(
            vocab_size=target_vocab_size,
            embedding_dim=embedding_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
        )

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
        target_input_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        Run the full encoder-decoder model.

        Parameters
        ----------
        source_ids : torch.Tensor
            Source token IDs of shape (batch_size, source_length).
        source_lengths : torch.Tensor
            Original source lengths before padding. Shape: (batch_size,).
        target_input_ids : torch.Tensor
            Target input IDs of shape (batch_size, target_length).

        Returns
        -------
        torch.Tensor
            Logits of shape (batch_size, target_length, target_vocab_size).
        """
        # Encode the source sentence.
        encoder_hidden = self.encoder(source_ids, source_lengths)

        # Decode the target input sequence using the encoder hidden state.
        logits, _ = self.decoder(target_input_ids, encoder_hidden)

        return logits

## 6. Training the RNN translator

The `TranslationDataset` returns three token ID sequences for each example:

```text
source_ids         = x_1, x_2, ..., x_T
target_input_ids  = <bos>, y_1, y_2, ..., y_S
target_output_ids = y_1,   y_2, ..., y_S, <eos>
```

The encoder receives `source_ids` and compresses the source sentence into a final hidden state.

The decoder receives `target_input_ids` and predicts one distribution over the target vocabulary at each position. The loss is computed by comparing these predictions to `target_output_ids`.

Padding tokens are ignored in the loss because they do not correspond to real target words.

### TODO: Study the training loop

Notice the loss function definition:

```python
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
```

Our mini-batches contain `<pad>` tokens because source and target sequences have different lengths. These padding tokens are not real words, so the model should not be rewarded or penalized for predictions at padded positions.

The argument `ignore_index=PAD_ID` tells PyTorch to ignore all target positions where the correct token is `<pad>` when computing the loss.

In [34]:
source_vocab_size = len(source_token_to_id)
target_vocab_size = len(target_token_to_id)

rnn_model = PlainRNNSeq2Seq(
    source_vocab_size=source_vocab_size,
    target_vocab_size=target_vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.0,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=1e-3)

num_epochs = 20

for epoch in range(num_epochs):
    rnn_model.train()

    total_train_loss = 0.0
    num_train_batches = 0

    for batch in train_loader:
        source_ids = batch["source_ids"].to(device)
        source_lengths = batch["source_lengths"].to(device)
        target_input_ids = batch["target_input_ids"].to(device)
        target_output_ids = batch["target_output_ids"].to(device)

        logits = rnn_model(
            source_ids=source_ids,
            source_lengths=source_lengths,
            target_input_ids=target_input_ids,
        )

        loss = criterion(
            logits.reshape(-1, target_vocab_size),
            target_output_ids.reshape(-1),
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        num_train_batches += 1

    average_train_loss = total_train_loss / num_train_batches

    rnn_model.eval()

    total_test_loss = 0.0
    num_test_batches = 0

    with torch.no_grad():
        for batch in test_loader:
            source_ids = batch["source_ids"].to(device)
            source_lengths = batch["source_lengths"].to(device)
            target_input_ids = batch["target_input_ids"].to(device)
            target_output_ids = batch["target_output_ids"].to(device)

            logits = rnn_model(
                source_ids=source_ids,
                source_lengths=source_lengths,
                target_input_ids=target_input_ids,
            )

            loss = criterion(
                logits.reshape(-1, target_vocab_size),
                target_output_ids.reshape(-1),
            )

            total_test_loss += loss.item()
            num_test_batches += 1

    average_test_loss = total_test_loss / num_test_batches

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"train loss: {average_train_loss:.4f} | "
        f"test loss: {average_test_loss:.4f}"
    )

Epoch 01/20 | train loss: 6.1523 | test loss: 5.2115
Epoch 02/20 | train loss: 5.2814 | test loss: 4.8098
Epoch 03/20 | train loss: 4.8984 | test loss: 4.5565
Epoch 04/20 | train loss: 4.6597 | test loss: 4.4274
Epoch 05/20 | train loss: 4.4963 | test loss: 4.3577
Epoch 06/20 | train loss: 4.3740 | test loss: 4.3057
Epoch 07/20 | train loss: 4.2695 | test loss: 4.2662
Epoch 08/20 | train loss: 4.1753 | test loss: 4.2375
Epoch 09/20 | train loss: 4.0894 | test loss: 4.2221
Epoch 10/20 | train loss: 4.0132 | test loss: 4.2077
Epoch 11/20 | train loss: 3.9396 | test loss: 4.1977
Epoch 12/20 | train loss: 3.8735 | test loss: 4.1924
Epoch 13/20 | train loss: 3.8079 | test loss: 4.1857
Epoch 14/20 | train loss: 3.7491 | test loss: 4.1949
Epoch 15/20 | train loss: 3.6893 | test loss: 4.1922
Epoch 16/20 | train loss: 3.6329 | test loss: 4.2071
Epoch 17/20 | train loss: 3.5808 | test loss: 4.2157
Epoch 18/20 | train loss: 3.5351 | test loss: 4.2218
Epoch 19/20 | train loss: 3.4820 | test loss: 

## 7. Greedy decoding with the RNN

During training, the decoder receives the target input sequence from the dataset:

```text
<bos>, y_1, y_2, ..., y_S
```

During inference, the true target sentence is not available. The model must generate the translation one token at a time.

We start the decoder with `<bos>`. Then, at each step:

1. the decoder predicts a distribution over the target vocabulary;
2. we select the token with the highest score;
3. the selected token is fed back into the decoder as the next input.

This is called **greedy decoding**.

Because the model uses its own previous predictions as inputs, errors can accumulate. If the model predicts a wrong token early, the next predictions are conditioned on that wrong token.

In [35]:
rnn_model.eval()

sample = test_dataset[0]

source_ids = sample["source_ids"].unsqueeze(0).to(device)
source_lengths = torch.tensor([sample["source_ids"].numel()], dtype=torch.long).to(
    device
)

generated_ids = [BOS_ID]

with torch.no_grad():
    # Encode the source sentence once.
    hidden = rnn_model.encoder(
        source_ids=source_ids,
        source_lengths=source_lengths,
    )

    # Start decoding from <bos>.
    current_token = torch.tensor([[BOS_ID]], dtype=torch.long).to(device)

    for step in range(20):
        logits, hidden = rnn_model.decoder(
            target_input_ids=current_token,
            hidden=hidden,
        )

        next_token_id = logits[:, -1, :].argmax(dim=-1).item()
        generated_ids.append(next_token_id)

        if next_token_id == EOS_ID:
            break

        current_token = torch.tensor([[next_token_id]], dtype=torch.long).to(device)

print("Source:")
print(sample["source_text"])

print("\nReference translation:")
print(sample["target_text"])

print("\nGenerated token IDs:")
print(generated_ids)

print("\nGenerated tokens:")
print(ids_to_tokens(generated_ids, target_id_to_token))

print("\nGenerated translation:")
print(tokens_to_text(ids_to_tokens(generated_ids, target_id_to_token)))

Source:
He was not listened to, and he had to prevent the pale and cowardly Pierron from entering among the first.

Reference translation:
On ne l'écoutait pas, il avait empeché Pierron, lâche et bleme, de filer un des premiers.

Generated token IDs:
[2, 13, 51, 97, 1, 4, 9, 13, 33, 1, 4, 9, 13, 33, 1, 4, 9, 13, 33, 1, 4]

Generated tokens:
['<bos>', 'il', 'avait', 'été', '<unk>', ',', 'et', 'il', 'se', '<unk>', ',', 'et', 'il', 'se', '<unk>', ',', 'et', 'il', 'se', '<unk>', ',']

Generated translation:
il avait été <unk> , et il se <unk> , et il se <unk> , et il se <unk> ,


The generated translation is poor, and this is expected. We are training a small word-level RNN encoder-decoder on only a small subset of the dataset. Many words are mapped to `<unk>` because they are rare or outside the limited vocabulary, and the plain RNN must compress the whole English sentence into a single hidden vector before decoding. During greedy decoding, early mistakes are also fed back into the decoder, so errors can quickly accumulate. This example should be interpreted as a demonstration of the sequence-to-sequence mechanism, not as a high-quality translation system.

## 8. Transformer encoder-decoder

<p align="left">
  <img 
    src="https://raw.githubusercontent.com/HannahPinson/tue-deeplearning-2AMM10/main/img/transformer.png" 
    alt="Transformer encoder-decoder architecture" 
    height="560"
  />
</p> 

A Transformer encoder-decoder follows the same sequence-to-sequence idea as the RNN model, but it does not process tokens recurrently. Instead, it uses attention.

The encoder receives the full source sequence and computes contextual representations of the source tokens. Unlike the plain RNN encoder, it does not compress the whole source sentence into a single final hidden state.

The decoder receives the target input sequence:

```text
<bos>, y_1, y_2, ..., y_S
```

and predicts the target output sequence:

```text
y_1, y_2, ..., y_S, <eos>
```

The Transformer decoder uses:

1. **masked self-attention** over the target input tokens;
2. **cross-attention** over the encoder source representations;
3. feed-forward layers applied independently to each token representation.

During training, the full target input sequence is available. Therefore, the decoder needs a **causal mask** so that each target position can only attend to itself and previous target positions, not future target positions.

Because the Transformer has no recurrence, it has no built-in notion of token order. We therefore add **positional encodings** to the token embeddings.

### Understanding the masked self-attention in the decoder

The Transformer decoder uses **masked self-attention** over the target input sequence.

<p align="left">
  <img 
    src="https://raw.githubusercontent.com/HannahPinson/tue-deeplearning-2AMM10/main/img/masked_sa.png" 
    alt="Masked self-attention in the Transformer decoder" 
    height="560"
  />
</p>

The target input sequence is shifted right:

```text
target_input_ids  = <bos>, y_1, y_2, ..., y_S
target_output_ids = y_1,   y_2, ..., y_S, <eos>
```

At decoder position \(t\), the model predicts the token at position \(t\) in `target_output_ids`. The input at the same position comes from `target_input_ids`, which contains the previous target token because of the shift.

Masked self-attention allows decoder position \(t\) to attend only to positions \(1, 2, ..., t\) in the target input sequence. It cannot attend to future positions \(t+1, t+2, ...\). This prevents the model from seeing future target tokens during training.

In PyTorch, this is implemented with a **causal mask**:

```python
target_length = target_input_ids.shape[1]

target_mask = nn.Transformer.generate_square_subsequent_mask(
    target_length
).to(device)
```

We also need a padding mask so that the model ignores `<pad>` tokens:

```python
target_padding_mask = target_input_ids == PAD_ID
```

So the decoder uses two different masks:

```text
causal mask  : prevents looking at future target positions
padding mask : prevents attending to <pad> tokens
```

### TODO: Implementing the transformer encoder-decoder architecture

In this section, you will complete the mask construction inside the Transformer model.

All masks in this implementation are **boolean masks**:

```text
False = attention is allowed
True  = attention is blocked
```

You will implement three masks:

```text
source_padding_mask : blocks attention to <pad> tokens in the source sequence
target_padding_mask : blocks attention to <pad> tokens in the target input sequence
target_causal_mask  : blocks attention to future target positions
```

The padding masks are created by comparing token IDs with `PAD_ID`.

The causal mask should be an upper-triangular matrix of shape:

```text
(target_length, target_length)
```

with `True` values above the main diagonal.

In [41]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding for Transformer models.

    Transformers do not process tokens recurrently, so token embeddings do not
    contain information about token order by themselves. Positional encodings
    are added to token embeddings to give the model information about the
    position of each token in the sequence.

    Parameters
    ----------
    embedding_dim : int
        Dimension of the token embeddings.
    max_length : int, default=5000
        Maximum supported sequence length.
    """

    def __init__(
        self,
        embedding_dim: int,
        max_length: int = 5000,
    ) -> None:
        super().__init__()

        positions = torch.arange(max_length).unsqueeze(1)

        div_terms = torch.exp(
            torch.arange(0, embedding_dim, 2) * (-math.log(10000.0) / embedding_dim)
        )

        positional_encoding = torch.zeros(max_length, embedding_dim)
        positional_encoding[:, 0::2] = torch.sin(positions * div_terms)
        positional_encoding[:, 1::2] = torch.cos(positions * div_terms)

        # Shape: (1, max_length, embedding_dim)
        # The first dimension is 1 so that it can broadcast over the batch.
        positional_encoding = positional_encoding.unsqueeze(0)

        self.register_buffer("positional_encoding", positional_encoding)

    def forward(self, token_embeddings: torch.Tensor) -> torch.Tensor:
        """
        Add positional encodings to token embeddings.

        Parameters
        ----------
        token_embeddings : torch.Tensor
            Token embeddings of shape
            (batch_size, sequence_length, embedding_dim).

        Returns
        -------
        torch.Tensor
            Token embeddings plus positional encodings.
            Shape: (batch_size, sequence_length, embedding_dim).
        """
        sequence_length = token_embeddings.shape[1]

        return token_embeddings + self.positional_encoding[:, :sequence_length, :]


class TransformerSeq2Seq(nn.Module):
    """
    Transformer encoder-decoder model for machine translation.

    This model uses PyTorch's built-in `nn.Transformer`. It receives source token
    IDs and target input token IDs, and predicts one distribution over the target
    vocabulary at each target position.

    In this implementation, all masks are boolean masks:

        False = attention is allowed
        True  = attention is blocked

    Parameters
    ----------
    source_vocab_size : int
        Number of tokens in the source vocabulary.
    target_vocab_size : int
        Number of tokens in the target vocabulary.
    embedding_dim : int, default=128
        Dimension of source and target token embeddings.
    hidden_dim : int, default=256
        Dimension of the feed-forward layers inside the Transformer.
    num_heads : int, default=4
        Number of attention heads.
    num_encoder_layers : int, default=2
        Number of Transformer encoder layers.
    num_decoder_layers : int, default=2
        Number of Transformer decoder layers.
    dropout : float, default=0.1
        Dropout probability.
    """

    def __init__(
        self,
        source_vocab_size: int,
        target_vocab_size: int,
        embedding_dim: int = 128,
        hidden_dim: int = 256,
        num_heads: int = 4,
        num_encoder_layers: int = 2,
        num_decoder_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.embedding_dim = embedding_dim

        self.source_embedding = nn.Embedding(
            num_embeddings=source_vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID,
        )

        self.target_embedding = nn.Embedding(
            num_embeddings=target_vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID,
        )

        self.positional_encoding = PositionalEncoding(
            embedding_dim=embedding_dim,
        )

        self.transformer = nn.Transformer(
            d_model=embedding_dim,
            nhead=num_heads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True,
        )

        self.output_layer = nn.Linear(
            in_features=embedding_dim,
            out_features=target_vocab_size,
        )

    def make_target_causal_mask(
        self,
        target_length: int,
        device: torch.device,
    ) -> torch.Tensor:
        """
        Create a boolean causal mask for decoder self-attention.

        The mask prevents each target position from attending to future target
        positions. A value of True means that attention is blocked.

        Parameters
        ----------
        target_length : int
            Length of the target input sequence.
        device : torch.device
            Device on which to create the mask.

        Returns
        -------
        torch.Tensor
            Boolean causal mask of shape (target_length, target_length).
        """
        # Create an upper-triangular boolean mask.
        # Shape: (target_length, target_length)
        #
        # False = attention is allowed
        # True  = attention is blocked
        #
        # The mask should be True above the main diagonal and False elsewhere.

        target_causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(target_length, device=device, dtype=torch.bool)
        return target_causal_mask

    def forward(
        self,
        source_ids: torch.Tensor,
        target_input_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        Run the Transformer encoder-decoder model.

        Parameters
        ----------
        source_ids : torch.Tensor
            Source token IDs of shape (batch_size, source_length).
        target_input_ids : torch.Tensor
            Target input token IDs of shape (batch_size, target_length).

        Returns
        -------
        torch.Tensor
            Logits of shape (batch_size, target_length, target_vocab_size).
        """
        # Create source padding mask.
        # Shape: (batch_size, source_length)
        # True values should mark source <pad> positions.
        source_padding_mask = (source_ids == PAD_ID)

        # Create target padding mask.
        # Shape: (batch_size, target_length)
        # True values should mark target <pad> positions.
        target_padding_mask = (target_input_ids == PAD_ID)

        target_length = target_input_ids.shape[1]

        # Create target causal mask using self.make_target_causal_mask.
        # Shape: (target_length, target_length)
        # True values should block attention to future target positions.
        target_causal_mask = self.make_target_causal_mask(target_length, device=target_input_ids.device)

        # Source token embeddings.
        # Shape: (batch_size, source_length, embedding_dim)
        source_embeddings = self.source_embedding(source_ids)

        # Target token embeddings.
        # Shape: (batch_size, target_length, embedding_dim)
        target_embeddings = self.target_embedding(target_input_ids)

        # Scale embeddings before adding positional encodings.
        source_embeddings = source_embeddings * math.sqrt(self.embedding_dim)
        target_embeddings = target_embeddings * math.sqrt(self.embedding_dim)

        # Add positional encodings.
        # Shapes stay the same:
        # source_embeddings: (batch_size, source_length, embedding_dim)
        # target_embeddings: (batch_size, target_length, embedding_dim)
        source_embeddings = self.positional_encoding(source_embeddings)
        target_embeddings = self.positional_encoding(target_embeddings)

        transformer_outputs = self.transformer(
            src=source_embeddings,  # Encoder input. Shape: (batch_size, source_length, embedding_dim)
            tgt=target_embeddings,  # Decoder input. Shape: (batch_size, target_length, embedding_dim)
            tgt_mask=target_causal_mask,  # Causal mask. Shape: (target_length, target_length)
            src_key_padding_mask=source_padding_mask,  # Source padding mask. Shape: (batch_size, source_length)
            tgt_key_padding_mask=target_padding_mask,  # Target padding mask. Shape: (batch_size, target_length)
            memory_key_padding_mask=source_padding_mask,  # Source padding mask for cross-attention. Shape: (batch_size, source_length)
        )

        # Transformer decoder outputs.
        # Shape: (batch_size, target_length, embedding_dim)

        # Project Transformer outputs to target vocabulary logits.
        # Shape should become:
        # (batch_size, target_length, target_vocab_size)
        logits = self.output_layer(transformer_outputs)

        return logits

### Training Loop

In [43]:
device = torch.device("cuda")

In [44]:
source_vocab_size = len(source_token_to_id)
target_vocab_size = len(target_token_to_id)

transformer_model = TransformerSeq2Seq(
    source_vocab_size=source_vocab_size,
    target_vocab_size=target_vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    num_heads=4,
    num_encoder_layers=1,
    num_decoder_layers=1,
    dropout=0.0,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-3)

num_epochs = 20

for epoch in range(num_epochs):
    transformer_model.train()

    total_train_loss = 0.0
    num_train_batches = 0

    for batch in train_loader:
        source_ids = batch["source_ids"].to(device)
        target_input_ids = batch["target_input_ids"].to(device)
        target_output_ids = batch["target_output_ids"].to(device)

        logits = transformer_model(
            source_ids=source_ids,
            target_input_ids=target_input_ids,
        )

        loss = criterion(
            logits.reshape(-1, target_vocab_size),
            target_output_ids.reshape(-1),
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        num_train_batches += 1

    average_train_loss = total_train_loss / num_train_batches

    transformer_model.eval()

    total_test_loss = 0.0
    num_test_batches = 0

    with torch.no_grad():
        for batch in test_loader:
            source_ids = batch["source_ids"].to(device)
            target_input_ids = batch["target_input_ids"].to(device)
            target_output_ids = batch["target_output_ids"].to(device)

            logits = transformer_model(
                source_ids=source_ids,
                target_input_ids=target_input_ids,
            )

            loss = criterion(
                logits.reshape(-1, target_vocab_size),
                target_output_ids.reshape(-1),
            )

            total_test_loss += loss.item()
            num_test_batches += 1

    average_test_loss = total_test_loss / num_test_batches

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"train loss: {average_train_loss:.4f} | "
        f"test loss: {average_test_loss:.4f}"
    )

c:\Users\trist\Documents\GitHub\tue-deeplearning-2AMM10\.venv\Lib\site-packages\torch\nn\functional.py:6487: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\transformers\hip\sdp_utils.cpp:361.)
  attn_output = scaled_dot_product_attention(
c:\Users\trist\Documents\GitHub\tue-deeplearning-2AMM10\.venv\Lib\site-packages\torch\nn\modules\transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\b\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(
c:\Users\trist\Documents\GitHub\tue-deeplearning-2AMM10\.venv\Lib\site-packages\torch

Epoch 01/20 | train loss: 6.2424 | test loss: 5.2019
Epoch 02/20 | train loss: 5.2218 | test loss: 4.7746
Epoch 03/20 | train loss: 4.8737 | test loss: 4.5934
Epoch 04/20 | train loss: 4.6561 | test loss: 4.4842
Epoch 05/20 | train loss: 4.4981 | test loss: 4.4290
Epoch 06/20 | train loss: 4.3662 | test loss: 4.3816
Epoch 07/20 | train loss: 4.2517 | test loss: 4.3484
Epoch 08/20 | train loss: 4.1490 | test loss: 4.3369
Epoch 09/20 | train loss: 4.0581 | test loss: 4.3398
Epoch 10/20 | train loss: 3.9717 | test loss: 4.3389
Epoch 11/20 | train loss: 3.8879 | test loss: 4.3526
Epoch 12/20 | train loss: 3.8094 | test loss: 4.3715
Epoch 13/20 | train loss: 3.7375 | test loss: 4.3934
Epoch 14/20 | train loss: 3.6665 | test loss: 4.4264
Epoch 15/20 | train loss: 3.5994 | test loss: 4.4487
Epoch 16/20 | train loss: 3.5351 | test loss: 4.4704
Epoch 17/20 | train loss: 3.4705 | test loss: 4.5047
Epoch 18/20 | train loss: 3.4056 | test loss: 4.5413
Epoch 19/20 | train loss: 3.3520 | test loss: 

## 9. Greedy decoding with the Transformer

During inference, the target sentence is not available. We start with `<bos>` and generate one token at a time.

Unlike the RNN decoder, the Transformer decoder does not keep a recurrent hidden state. Instead, at every step we pass the full generated prefix back into the model. The causal mask ensures that each position can only attend to previous positions in this prefix.

In [45]:
transformer_model.eval()

sample = test_dataset[0]

source_ids = sample["source_ids"].unsqueeze(0).to(device)

generated_ids = [BOS_ID]

with torch.no_grad():
    for step in range(20):
        target_input_ids = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device,
        )

        logits = transformer_model(
            source_ids=source_ids,
            target_input_ids=target_input_ids,
        )

        # Take the prediction from the last generated position.
        next_token_id = logits[:, -1, :].argmax(dim=-1).item()

        generated_ids.append(next_token_id)

        if next_token_id == EOS_ID:
            break

print("Source:")
print(sample["source_text"])

print("\nReference translation:")
print(sample["target_text"])

print("\nGenerated token IDs:")
print(generated_ids)

print("\nGenerated tokens:")
print(ids_to_tokens(generated_ids, target_id_to_token))

print("\nGenerated translation:")
print(tokens_to_text(ids_to_tokens(generated_ids, target_id_to_token)))

Source:
He was not listened to, and he had to prevent the pale and cowardly Pierron from entering among the first.

Reference translation:
On ne l'écoutait pas, il avait empeché Pierron, lâche et bleme, de filer un des premiers.

Generated token IDs:
[2, 13, 38, 1, 4, 9, 13, 642, 7, 15, 6, 38, 1, 9, 13, 5496, 14, 10, 202, 9, 13]

Generated tokens:
['<bos>', 'il', 'était', '<unk>', ',', 'et', 'il', 'tenait', 'de', 'l', "'", 'était', '<unk>', 'et', 'il', 'méditait', 'à', 'la', 'maison', 'et', 'il']

Generated translation:
il était <unk> , et il tenait de l ' était <unk> et il méditait à la maison et il
